# 👶 Baby Growth Journal — Gemini + MCP Agent

An agentic app for new parents: log your baby's measurements and photos, compare growth
against WHO reference averages, and generate a 6‑month update (collage + growth chart +
written summary) — all orchestrated by a Gemini agent that decides which tools to call.

**MCP servers used**
1. `filesystem` (third‑party, official) — read/write baby data & photos in Google Drive
2. `git` (third‑party, official) — every new measurement is committed, giving a full history log
3. `baby_ops` (custom, built with FastMCP) — growth‑percentile math, chart generation, photo collage, 6‑month report

**Note:** the WHO growth values embedded below are approximate 50th‑percentile (median)
reference points meant for a school project demo — they are **not** clinically precise
and should not be used for real medical decisions.


## 1. Install dependencies

In [ ]:
%pip install -qU \
  "langchain>=0.3" \
  "langgraph>=0.2" \
  "langchain-google-genai>=2.0" \
  "google-genai>=1.0" \
  "langchain-mcp-adapters==0.2.1" \
  "nest_asyncio" \
  "fastmcp>=2.0.0" \
  "matplotlib" \
  "pillow"


## 2. Set your Gemini API key

In [ ]:
import os
from getpass import getpass

if "GOOGLE_API_KEY" not in os.environ or not os.environ["GOOGLE_API_KEY"]:
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your GOOGLE_API_KEY: ")
print("GOOGLE_API_KEY is set:", bool(os.environ.get("GOOGLE_API_KEY")))


## 3. Confirm Node/NPM availability (needed for the filesystem MCP server)

In [ ]:
!node --version
!npx --version


In [ ]:
# Run this cell only if the previous cell errored out
!apt-get -qq update
!apt-get -qq install -y nodejs npm
!node --version
!npx --version


## 4. Mount Google Drive and set up the working folder

Photos and baby data live in your Drive so they persist across Colab sessions.
Put any photos you want included in updates into the `photos/` subfolder.


In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

WORKDIR = "/content/drive/MyDrive/BabyGrowthJournal"
PHOTOS_DIR = os.path.join(WORKDIR, "photos")
DATA_FILE = os.path.join(WORKDIR, "baby_data.json")
REPORTS_DIR = os.path.join(WORKDIR, "reports")

os.makedirs(WORKDIR, exist_ok=True)
os.makedirs(PHOTOS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

if not os.path.exists(DATA_FILE):
    with open(DATA_FILE, "w") as f:
        f.write("[]")

print("WORKDIR:", WORKDIR)
print("Photos folder (drop baby photos here):", PHOTOS_DIR)


## 5. Initialize WORKDIR as a git repo (for the git MCP server)

In [ ]:
import subprocess

def run(cmd, cwd=WORKDIR):
    result = subprocess.run(cmd, cwd=cwd, shell=True, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)

if not os.path.exists(os.path.join(WORKDIR, ".git")):
    run("git init")
    run('git config user.email "parent@babygrowthjournal.local"')
    run('git config user.name "Baby Growth Journal"')
    run("git add -A")
    run('git commit -m "Initial commit: baby growth journal setup"')
else:
    print("Repo already initialized.")


## 6. Build the custom `baby_ops` MCP server (FastMCP)

Tools:
- `add_measurement` — log a new measurement (appends to `baby_data.json`)
- `get_growth_percentile` — compare a measurement against WHO median reference curves
- `generate_growth_chart` — plot the baby's tracked history vs. the WHO average curve
- `create_collage` — combine all photos in the photos folder into one collage image
- `six_month_update` — one-call tool producing chart + collage + written summary


In [ ]:
from pathlib import Path
import textwrap

server_path = Path("/content/baby_ops_server.py")
server_path.write_text(textwrap.dedent(f"""
    from fastmcp import FastMCP
    from typing import Optional
    import json, os
    from datetime import datetime

    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from PIL import Image

    mcp = FastMCP(name="baby_ops")

    WORKDIR = {WORKDIR!r}
    PHOTOS_DIR = {PHOTOS_DIR!r}
    DATA_FILE = {DATA_FILE!r}
    REPORTS_DIR = {REPORTS_DIR!r}

    # Approximate WHO Child Growth Standards, 50th percentile (median) reference points.
    # age_months -> value. For demo purposes only, not clinically precise.
    WHO_WEIGHT_KG = {{
        "boy":  {{0:3.3,1:4.5,2:5.6,3:6.4,4:7.0,5:7.5,6:7.9,7:8.3,8:8.6,9:8.9,10:9.2,11:9.4,
                  12:9.6,15:10.3,18:10.9,21:11.5,24:12.2}},
        "girl": {{0:3.2,1:4.2,2:5.1,3:5.8,4:6.4,5:6.9,6:7.3,7:7.6,8:7.9,9:8.2,10:8.5,11:8.7,
                  12:8.9,15:9.6,18:10.2,21:10.9,24:11.5}},
    }}
    WHO_HEIGHT_CM = {{
        "boy":  {{0:49.9,1:54.7,2:58.4,3:61.4,4:63.9,5:65.9,6:67.6,7:69.2,8:70.6,9:72.0,
                  10:73.3,11:74.5,12:75.7,15:78.6,18:81.2,21:83.7,24:87.1}},
        "girl": {{0:49.1,1:53.7,2:57.1,3:59.8,4:62.1,5:64.0,6:65.7,7:67.3,8:68.7,9:70.1,
                  10:71.5,11:72.8,12:74.0,15:76.8,18:79.3,21:81.7,24:85.0}},
    }}

    def _interp(table, age_months):
        keys = sorted(table.keys())
        if age_months <= keys[0]:
            return table[keys[0]]
        if age_months >= keys[-1]:
            return table[keys[-1]]
        for lo, hi in zip(keys, keys[1:]):
            if lo <= age_months <= hi:
                frac = (age_months - lo) / (hi - lo)
                return table[lo] + frac * (table[hi] - table[lo])
        return table[keys[-1]]

    def _load():
        with open(DATA_FILE) as f:
            return json.load(f)

    def _save(records):
        with open(DATA_FILE, "w") as f:
            json.dump(records, f, indent=2)

    @mcp.tool
    def ping() -> str:
        \"\"\"Health check tool.\"\"\"
        return "pong"

    @mcp.tool
    def add_measurement(date: str, age_months: float, weight_kg: float, height_cm: float,
                         head_circumference_cm: Optional[float] = None, sex: str = "boy",
                         note: str = "") -> dict:
        \"\"\"Log a new baby measurement. date is YYYY-MM-DD. sex is 'boy' or 'girl'.\"\"\"
        records = _load()
        entry = {{
            "date": date, "age_months": age_months, "weight_kg": weight_kg,
            "height_cm": height_cm, "head_circumference_cm": head_circumference_cm,
            "sex": sex, "note": note,
        }}
        records.append(entry)
        _save(records)
        return {{"status": "saved", "entry": entry, "total_records": len(records)}}

    @mcp.tool
    def list_measurements() -> list:
        \"\"\"Return all logged measurements.\"\"\"
        return _load()

    @mcp.tool
    def get_growth_percentile(age_months: float, weight_kg: float, height_cm: float,
                               sex: str = "boy") -> dict:
        \"\"\"Compare a measurement to the WHO median reference for that age and sex.\"\"\"
        sex = sex if sex in ("boy", "girl") else "boy"
        ref_w = _interp(WHO_WEIGHT_KG[sex], age_months)
        ref_h = _interp(WHO_HEIGHT_CM[sex], age_months)
        return {{
            "age_months": age_months, "sex": sex,
            "weight_kg": weight_kg, "who_median_weight_kg": round(ref_w, 2),
            "weight_diff_kg": round(weight_kg - ref_w, 2),
            "height_cm": height_cm, "who_median_height_cm": round(ref_h, 2),
            "height_diff_cm": round(height_cm - ref_h, 2),
        }}

    @mcp.tool
    def generate_growth_chart(sex: str = "boy") -> str:
        \"\"\"Plot the baby's tracked weight & height history vs. the WHO median curve.
        Returns the saved chart file path.\"\"\"
        sex = sex if sex in ("boy", "girl") else "boy"
        records = sorted(_load(), key=lambda r: r["age_months"])
        if not records:
            return "No measurements logged yet."

        ages = [r["age_months"] for r in records]
        weights = [r["weight_kg"] for r in records]
        heights = [r["height_cm"] for r in records]

        who_ages = sorted(WHO_WEIGHT_KG[sex].keys())
        who_w = [WHO_WEIGHT_KG[sex][a] for a in who_ages]
        who_h = [WHO_HEIGHT_CM[sex][a] for a in who_ages]

        fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

        axes[0].plot(who_ages, who_w, "--", color="gray", label="WHO median")
        axes[0].plot(ages, weights, "o-", color="#2563eb", label="Your baby")
        axes[0].set_title("Weight vs. age")
        axes[0].set_xlabel("Age (months)"); axes[0].set_ylabel("Weight (kg)")
        axes[0].legend()

        axes[1].plot(who_ages, who_h, "--", color="gray", label="WHO median")
        axes[1].plot(ages, heights, "o-", color="#16a34a", label="Your baby")
        axes[1].set_title("Height vs. age")
        axes[1].set_xlabel("Age (months)"); axes[1].set_ylabel("Height (cm)")
        axes[1].legend()

        fig.suptitle(f"Growth chart ({{sex}})")
        fig.tight_layout()
        out_path = os.path.join(REPORTS_DIR, "growth_chart.png")
        fig.savefig(out_path, dpi=150)
        plt.close(fig)
        return out_path

    @mcp.tool
    def create_collage(output_name: str = "photo_collage.png") -> str:
        \"\"\"Combine every photo in the photos folder into a single grid collage image.
        Returns the saved collage file path.\"\"\"
        exts = (".png", ".jpg", ".jpeg", ".webp")
        photo_files = sorted(
            [os.path.join(PHOTOS_DIR, f) for f in os.listdir(PHOTOS_DIR) if f.lower().endswith(exts)]
        )
        if not photo_files:
            return "No photos found in the photos folder."

        thumbs = []
        thumb_size = (400, 400)
        for p in photo_files:
            try:
                img = Image.open(p).convert("RGB")
                img.thumbnail(thumb_size)
                canvas = Image.new("RGB", thumb_size, (245, 245, 245))
                offset = ((thumb_size[0] - img.width) // 2, (thumb_size[1] - img.height) // 2)
                canvas.paste(img, offset)
                thumbs.append(canvas)
            except Exception:
                continue

        if not thumbs:
            return "Could not open any photos."

        n = len(thumbs)
        cols = min(4, n)
        rows = (n + cols - 1) // cols
        collage = Image.new("RGB", (cols * thumb_size[0], rows * thumb_size[1]), (255, 255, 255))
        for i, t in enumerate(thumbs):
            x = (i % cols) * thumb_size[0]
            y = (i // cols) * thumb_size[1]
            collage.paste(t, (x, y))

        out_path = os.path.join(REPORTS_DIR, output_name)
        collage.save(out_path)
        return out_path

    @mcp.tool
    def six_month_update(sex: str = "boy") -> dict:
        \"\"\"Produce a combined 6-month update: growth chart + photo collage + a written summary
        of how the baby's stats compare to the WHO median.\"\"\"
        records = sorted(_load(), key=lambda r: r["age_months"])
        chart_path = generate_growth_chart(sex)
        collage_path = create_collage("six_month_collage.png")

        summary_lines = [f"6-Month Update — generated {{datetime.now().strftime('%Y-%m-%d')}}", ""]
        if records:
            latest = records[-1]
            comp = get_growth_percentile(latest["age_months"], latest["weight_kg"],
                                          latest["height_cm"], sex)
            summary_lines.append(
                f"Latest measurement ({{latest['date']}}, age {{latest['age_months']}} mo): "
                f"weight {{latest['weight_kg']}}kg (WHO median {{comp['who_median_weight_kg']}}kg, "
                f"diff {{comp['weight_diff_kg']:+}}kg), "
                f"height {{latest['height_cm']}}cm (WHO median {{comp['who_median_height_cm']}}cm, "
                f"diff {{comp['height_diff_cm']:+}}cm)."
            )
            notes = [r["note"] for r in records if r.get("note")]
            if notes:
                summary_lines.append("Milestones/notes logged: " + "; ".join(notes))
        else:
            summary_lines.append("No measurements logged yet.")

        summary_text = "\\n".join(summary_lines)
        summary_path = os.path.join(REPORTS_DIR, "six_month_summary.txt")
        with open(summary_path, "w") as f:
            f.write(summary_text)

        return {{
            "chart_path": chart_path,
            "collage_path": collage_path,
            "summary_path": summary_path,
            "summary_text": summary_text,
        }}

    if __name__ == "__main__":
        mcp.run(transport="stdio")
"""), encoding="utf-8")

print("Wrote custom MCP server to:", server_path)


## 7. Connect to all three MCP servers

In [ ]:
import asyncio
import nest_asyncio
nest_asyncio.apply()

from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_connections = {
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", WORKDIR],
    },
    "git": {
        "transport": "stdio",
        "command": "python",
        "args": ["-m", "mcp_server_git", "--repository", WORKDIR],
    },
    "baby_ops": {
        "transport": "stdio",
        "command": "python",
        "args": [str(server_path)],
    },
}

client = MultiServerMCPClient(mcp_connections)
tools = asyncio.get_event_loop().run_until_complete(client.get_tools())

print("Tool count:", len(tools))
print([t.name for t in tools])


## 8. Build the Gemini agent

The agent decides which MCP tools to call and in what order — nothing here hard-codes
"log, then chart, then collage." Ask it naturally and it plans the steps itself.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

SYSTEM_PROMPT = """You are a helpful baby growth journal assistant.
You have tools to: log measurements, read past measurements, compare a measurement
to WHO median growth references, generate a growth chart, create a photo collage from
the photos folder, produce a full 6-month update, read/write files in the baby's
working folder, and commit changes to git so there is a history of what was known and when.

Whenever you log a new measurement, also commit the change to git with a short,
descriptive commit message so there is a version history.
Always mention that WHO comparisons are approximate and for informational purposes only,
not medical advice.
"""

agent = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)
print("Agent ready.")


## 9. Helper to chat with the agent

In [ ]:
async def ask_agent(query: str):
    result = await agent.ainvoke({"messages": [("user", query)]})
    final_message = result["messages"][-1]
    print(final_message.content)
    return result

def ask(query: str):
    return asyncio.get_event_loop().run_until_complete(ask_agent(query))


## 10. Demo — log a measurement

In [ ]:
ask(
    "Log a new measurement for today: age 3 months, weight 6.2kg, height 60cm, "
    "sex boy, note 'smiled for the first time'. Save it and commit it to git."
)


## 11. Demo — compare to WHO average

In [ ]:
ask(
    "How does my baby's latest weight and height compare to the WHO average for "
    "a 3 month old boy? Generate a growth chart too."
)


## 12. Demo — 6-month update

Make sure you've dropped some photos into the `photos/` folder in your mounted Drive
(`BabyGrowthJournal/photos`) before running this.


In [ ]:
ask(
    "Generate the 6 month update: create a photo collage from the photos folder, "
    "build the growth chart, and give me a written summary. Commit the results to git."
)


## 13. View the generated files

Run this to preview the chart and collage saved into `reports/`.


In [ ]:
from IPython.display import Image as IPImage, display
import os

for fname in ["growth_chart.png", "six_month_collage.png"]:
    fpath = os.path.join(REPORTS_DIR, fname)
    if os.path.exists(fpath):
        print(fname)
        display(IPImage(filename=fpath))
    else:
        print(fname, "not generated yet.")


## Notes & limitations

- WHO reference values embedded in `baby_ops_server.py` are approximate 50th-percentile
  medians for a school project demo, not full LMS percentile tables — do not use for real
  medical decisions.
- Photos must be placed in `BabyGrowthJournal/photos/` inside your Google Drive before
  running the collage/6-month-update demos.
- Every measurement commit is stored in the git history inside `BabyGrowthJournal/`, so you
  can always go back and see what was known about the baby at any earlier point in time.
